In [1]:

import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, timeout=600)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))


c:\Users\jorge\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-06-23 21:17:46,084] A new study created in memory with name: no-name-4a2b5548-679c-49dc-afd9-d8cda9155cb0
[I 2024-06-23 21:25:40,709] Trial 0 finished with value: 0.9196261682242991 and parameters: {'dropout': 0.5023933392789214, 'optimizer': 'SGD', 'lr': 0.0026754897888767596}. Best is trial 0 with value: 0.9196261682242991.
[I 2024-06-23 21:33:15,910] Trial 1 finished with value: 0.914018691588785 and parameters: {'dropout': 0.1530575566819227, 'optimizer': 'RMSprop', 'lr': 0.03756863773848415}. Best is trial 0 with value: 0.9196261682242991.


Study statistics: 
  Number of finished trials:  2
  Number of pruned trials:  0
  Number of complete trials:  2
Best trial:
  Value:  0.9196261682242991
  Params: 
    dropout: 0.5023933392789214
    optimizer: SGD
    lr: 0.0026754897888767596


In [2]:

import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 1024), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(dropout),
       nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(128, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, timeout=600)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))


[I 2024-06-23 21:33:17,847] A new study created in memory with name: no-name-6fddfb82-536c-40cb-a2ff-4f78ef0e1ea6
[I 2024-06-23 21:40:54,468] Trial 0 finished with value: 0.8934579439252337 and parameters: {'dropout': 0.34497292165034776, 'optimizer': 'Adam', 'lr': 0.016625763204286496}. Best is trial 0 with value: 0.8934579439252337.
[I 2024-06-23 21:48:30,137] Trial 1 finished with value: 0.9177570093457944 and parameters: {'dropout': 0.45170516825654505, 'optimizer': 'Adam', 'lr': 2.6985938845312775e-05}. Best is trial 1 with value: 0.9177570093457944.


Study statistics: 
  Number of finished trials:  2
  Number of pruned trials:  0
  Number of complete trials:  2
Best trial:
  Value:  0.9177570093457944
  Params: 
    dropout: 0.45170516825654505
    optimizer: Adam
    lr: 2.6985938845312775e-05


## Multiplicando la data

In [1]:

import os
import numpy as np
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

DG_folder0='data_augx4AB'
images_increased = 4

try:
    os.mkdir(DG_folder0)
except:
    print("")
    
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False)


data_path = r"Art_Brut" 
data_dir_list = os.listdir(data_path)

width_shape, height_shape = 224, 224

i=0
num_images=0
for image_file in data_dir_list:
    img_list=os.listdir(data_path)

    img_path = data_path + '/'+ image_file

    imge=load_img(img_path)
    
    imge=cv2.resize(img_to_array(imge), (width_shape, height_shape), interpolation = cv2.INTER_AREA)
    x= imge/255
    x=np.expand_dims(x,axis=0)
    t=1
    for output_batch in train_datagen.flow(x,batch_size=1):
        a=img_to_array(output_batch[0])
        imagen=output_batch[0,:,:]*255
        imgfinal = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        cv2.imwrite(DG_folder0+"/%i%i.jpg"%(i,t), imgfinal) 
        t+=1
        
        num_images+=1
        if t>images_increased:
            break
    i+=1
    
print("images generated",num_images)


images generated 1900


In [3]:
import os
import numpy as np
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

DG_folder1='data_augx4Re'
images_increased = 4

try:
    os.mkdir(DG_folder1)
except:
    print("")
    
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False)


data_path = r"Realism/"
data_dir_list = os.listdir(data_path)

width_shape, height_shape = 224, 224

i=0
num_images=0
for image_file in data_dir_list:
    img_list=os.listdir(data_path)

    img_path = data_path + '/'+ image_file
    try:
        imge=load_img(img_path)
        
        imge=cv2.resize(img_to_array(imge), (width_shape, height_shape), interpolation = cv2.INTER_AREA)
        x= imge/255
        x=np.expand_dims(x,axis=0)
        t=1
        for output_batch in train_datagen.flow(x,batch_size=1):
            a=img_to_array(output_batch[0])
            imagen=output_batch[0,:,:]*255
            imgfinal = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
            cv2.imwrite(DG_folder1+"/%i%i.jpg"%(i,t), imgfinal) 
            t+=1
            
            num_images+=1
            if t>images_increased:
                break
        i+=1
    except PermissionError as e:
        print(e)
    
print("images generated",num_images)


images generated 1836


In [4]:
import os
import random
import shutil
from skimage.io import imread_collection
from torch.utils.data import random_split

imagenes_brut = imread_collection('data_augx4AB/*.jpg')
imagenes_realism = imread_collection('data_augx4Re/*.jpg')

#directorios donde se encuentran las imagenes 
directorio_brut = "data_augx4AB/"
directorio_realism = "data_augx4Re/"

carpeta = ['train','test','val']
estilo = ['brut','realism']
#dIRECTORIOS DE SALIDA
directorios =[]
for i in range(3):
    for j in range(2):
        nombre_directorio = f"directorio_{carpeta[i]}_{estilo[j]}"
        x = os.path.join ("data4",carpeta[i],estilo[j])
        directorios.append(x)

In [5]:

for directorio in directorios:
    if not os.path.exists(directorio):
        os.makedirs(directorio)

im_files_brut = os.listdir(directorio_brut)
im_files_realism = os.listdir(directorio_realism)

# Calcula el tamaño para cada conjunto
def tamaño_conjunto(dataset):
    train_ratio = 0.6
    val_ratio = 0.3
    test_ratio = 0.1
    total_size = len(dataset)
    train_size = int(train_ratio * total_size)
    val_size = int(val_ratio * total_size)
    test_size = total_size - train_size - val_size
    return(total_size,train_size,test_size,val_size)

n,train_size,test_size,val_size = tamaño_conjunto(imagenes_realism)

#m,train_size,test_size,val_size = tamaño_conjunto(imagenes_brut)

im_files_brut = random.sample(im_files_brut, n)
im_files_realism = random.sample(im_files_realism, n)

imagenes_train_brut, rest_dataset1 = random_split(im_files_brut, [train_size, n - train_size])
imagenes_val_brut, imagenes_test_brut = random_split(rest_dataset1, [val_size, test_size])

imagenes_train_realism, rest_dataset2 = random_split(im_files_realism, [train_size, n - train_size])
imagenes_val_realism, imagenes_test_realism = random_split(rest_dataset2, [val_size, test_size])

imagenes = [imagenes_train_brut,imagenes_train_realism,imagenes_test_brut,imagenes_test_realism,imagenes_val_brut,imagenes_val_realism]

for i,lista in enumerate(imagenes):
    if i%2==0:
        print(i)
        for j in lista:
            #print(directorios[i],lista,'par realism')
            ruta_origen = os.path.join(directorio_realism,j)
            ruta_destino = os.path.join(directorios[i+1],j)
            try:
                shutil.copy(ruta_origen,ruta_destino)
            except:
                print('error') 
    else:
        for k in lista:
            
            #print(directorios[i],lista,'impar, brut')
            ruta_origen = os.path.join(directorio_brut,k)
            ruta_destino = os.path.join(directorios[i-1], k)
            try:
                shutil.copy(ruta_origen,ruta_destino)
            except:
                print('error')

0
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
2
error
error
4
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error
error


In [6]:
import pandas as pd

save_file_name = 'vgg16-transfer-4.pt'
checkpoint_path = 'vgg16-transfer-4.pth'

# Directorio principal que contiene los datos
data_dir = 'data4/'

# Lista para almacenar la información de cada imagen
lista_train = []
lista_test = []
lista_val = []
# Iterar sobre las carpetas de train, test y validation
for split in ['train', 'test', 'val']:
    split_dir = os.path.join(data_dir, split)
    for clase in os.listdir(split_dir):
        clase_dir = os.path.join(split_dir, clase)
        for filename in os.listdir(clase_dir):
            # Generar la ruta completa del archivo
            path = os.path.join(clase_dir, filename)
            # Obtener el nombre de la clase y el índice de clase a partir del nombre del directorio
            class_index = 0 if clase == 'brut' else 1
            # Agregar la información de la imagen a la lista
            if split == 'train':
                lista_train.append((filename, path, clase, class_index))
            elif split == 'test':
                lista_test.append((filename, path, clase, class_index))
            else:
                lista_val.append((filename, path, clase, class_index))

# Crear el DataFrame
df_train = pd.DataFrame(lista_train, columns=['file_name', 'file_path', 'class_name', 'class_index'])
df_test = pd.DataFrame(lista_test, columns=['file_name', 'file_path', 'class_name', 'class_index'])
df_val = pd.DataFrame(lista_val, columns=['file_name', 'file_path', 'class_name', 'class_index'])
# Guardar el DataFrame en un archivo CSV
df_train.to_csv('etiquetas_train4.csv', index=False)
df_test.to_csv('etiquetas_test4.csv', index=False)
df_val.to_csv('etiquetas_val4.csv', index=False)


print("CSV generado exitosamente.")

CSV generado exitosamente.


## x4

#### Prueba 1 
2 c lineales

In [7]:
import os
import optuna
import optuna_dashboard
from optuna_dashboard import run_server
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy

if __name__ == "__main__":
    storage = optuna.storages.InMemoryStorage()
    study = optuna.create_study(direction="maximize",storage=storage)
    study.optimize(objective, n_trials=100, timeout=600)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

    # Run the Optuna dashbord server   
    run_server(storage)

c:\Users\Abi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-06-25 14:28:01,404] A new study created in memory with name: no-name-caf24f1d-ff56-40bc-80a6-4cb846278ca9


#### prueba 2
3 c lineales


In [26]:

import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 1024), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(1024, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, timeout=600)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))


[I 2024-06-24 07:11:15,823] A new study created in memory with name: no-name-e43790b9-cd43-4083-a61a-86afb15f1fde
[I 2024-06-24 07:21:18,427] Trial 0 finished with value: 0.7822878228782287 and parameters: {'dropout': 0.4208996451926935, 'optimizer': 'SGD', 'lr': 0.00024615458100193255}. Best is trial 0 with value: 0.7822878228782287.


Study statistics: 
  Number of finished trials:  1
  Number of pruned trials:  0
  Number of complete trials:  1
Best trial:
  Value:  0.7822878228782287
  Params: 
    dropout: 0.4208996451926935
    optimizer: SGD
    lr: 0.00024615458100193255


#### Prueba 3
4 capas

In [27]:

import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 1024), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512,256),nn.ReLU(),nn.Dropout(dropout),
        nn.Linear(256, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, timeout=600)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

[I 2024-06-24 07:21:20,040] A new study created in memory with name: no-name-11cb2472-02a3-44f5-a21b-f3b9762b0614
[I 2024-06-24 07:31:22,832] Trial 0 finished with value: 0.492619926199262 and parameters: {'dropout': 0.42792963093085556, 'optimizer': 'Adam', 'lr': 0.08026285878546927}. Best is trial 0 with value: 0.492619926199262.


Study statistics: 
  Number of finished trials:  1
  Number of pruned trials:  0
  Number of complete trials:  1
Best trial:
  Value:  0.492619926199262
  Params: 
    dropout: 0.42792963093085556
    optimizer: Adam
    lr: 0.08026285878546927


#### prueba 4 timeout 600

In [28]:

import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 1024), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(128, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, timeout=600)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

[I 2024-06-24 07:31:24,273] A new study created in memory with name: no-name-82486163-80f6-4730-8545-185bcaab6bca
[I 2024-06-24 07:41:25,655] Trial 0 finished with value: 0.9621771217712177 and parameters: {'dropout': 0.36243592965559895, 'optimizer': 'RMSprop', 'lr': 0.00015686463733078726}. Best is trial 0 with value: 0.9621771217712177.


Study statistics: 
  Number of finished trials:  1
  Number of pruned trials:  0
  Number of complete trials:  1
Best trial:
  Value:  0.9621771217712177
  Params: 
    dropout: 0.36243592965559895
    optimizer: RMSprop
    lr: 0.00015686463733078726


#### Prueba 5, (4 + tiempo 5000)


In [29]:

import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms
import cv2
import pandas as pd
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
BATCHSIZE = 128
CLASSES = 2
DIR = os.getcwd()
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

import torch
import torch.nn as nn
import torchvision.models as models

# Carga la arquitectura VGG16 preentrenada
vgg16 = models.vgg16(weights = models.VGG16_Weights.IMAGENET1K_V1)

# Deshabilita el cálculo de gradientes para las capas existentes
for param in vgg16.parameters():
        param.requires_grad = False
#vgg16 = nn.Sequential(*list(vgg16.children())[:-1])
        
def define_model(trial,modelo = vgg16):
    dropout = trial.suggest_float("dropout", 0.1, 0.8, log=True)
    # Dimensiones iniciales
    #n_inputs = modelo.classifier[-1].in_features
    n_inputs = 25088
        # Add on classifier
    clasiffier_new = nn.Sequential(
        nn.Linear(n_inputs, 1024), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(128, 2), nn.LogSoftmax(dim=1)  ) 
    
    # Reemplaza la capa clasificadora en VGG16
    modelo.classifier = clasiffier_new
    return modelo

###############################################################################
class Dataset_imagenes(torch.utils.data.Dataset):
    def __init__(self,etiquetas,img_dir="",transforms=None):
        self.etiquetas_df = pd.read_csv(etiquetas,delimiter=',', header=0)
        self.img_dir = img_dir
        self.transform = transforms
        self.clase_a_indice = {
            'brut': 0,
            'realism': 1
        }

    def __len__(self):
        return len(self.etiquetas_df)
    
    def __getitem__(self, idx):
        self.loaded_data = []
        directorio_imagen = os.path.join(self.img_dir, self.etiquetas_df.iloc[idx, 1]) 
        imagen = cv2.imread(directorio_imagen) 
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        clase = self.etiquetas_df.iloc[idx, 2]
        clase_indice = self.clase_a_indice[clase]
        clase_tensor = torch.tensor(clase_indice, dtype=torch.long)
        if self.transform:
            imagen = self.transform(imagen)      
        return imagen, clase_tensor
    

from torchvision import transforms
from torch.utils.data import DataLoader

t0=transforms.Compose([transforms.ToTensor(),transforms.Resize((224,224)),transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) ])

set_datos_train = Dataset_imagenes(etiquetas ='etiquetas_train4.csv',img_dir="",transforms=t0)
set_datos_val = Dataset_imagenes(etiquetas ='etiquetas_val4.csv',img_dir="",transforms=t0) 

train_loader = DataLoader(set_datos_train, batch_size=BATCHSIZE, shuffle=True)
valid_loader = DataLoader(set_datos_val, batch_size=BATCHSIZE, shuffle=True)


from torchsummary import summary
###############################################################

def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    #train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            else:
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
            
            #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data = data.to(device=DEVICE, dtype=torch.float32)
                target = target.to(device=DEVICE, dtype=torch.long)
                #data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, timeout=5000)

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

[I 2024-06-24 07:41:28,434] A new study created in memory with name: no-name-b2487fad-0b82-4097-a8f2-1d2651072574
[I 2024-06-24 07:51:32,550] Trial 0 finished with value: 0.9547970479704797 and parameters: {'dropout': 0.4656946174776841, 'optimizer': 'Adam', 'lr': 2.6424281256635044e-05}. Best is trial 0 with value: 0.9547970479704797.
[I 2024-06-24 08:01:34,482] Trial 1 finished with value: 0.9658671586715867 and parameters: {'dropout': 0.2568578179220404, 'optimizer': 'Adam', 'lr': 0.0044124934644673335}. Best is trial 1 with value: 0.9658671586715867.
[I 2024-06-24 08:11:50,625] Trial 2 finished with value: 0.9695571955719557 and parameters: {'dropout': 0.19455884625310033, 'optimizer': 'Adam', 'lr': 0.00010656100084981636}. Best is trial 2 with value: 0.9695571955719557.
[I 2024-06-24 08:21:55,059] Trial 3 finished with value: 0.9630996309963099 and parameters: {'dropout': 0.14285518166667416, 'optimizer': 'Adam', 'lr': 0.0007787482500433817}. Best is trial 2 with value: 0.96955719

Study statistics: 
  Number of finished trials:  12
  Number of pruned trials:  6
  Number of complete trials:  6
Best trial:
  Value:  0.9695571955719557
  Params: 
    dropout: 0.19455884625310033
    optimizer: Adam
    lr: 0.00010656100084981636


Dashboard optuna...

In [1]:
pip install optuna-dashboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1
[notice] To update, run: python.exe -m pip install --upgrade pip
